# LeKiwi 底盘手动试车 / Manual base test

点击每个格子运行，看轮子/底盘怎么动。每次动作都是**有界的**：跑固定时长后自动停、松扭矩，不会失控。

## ⚠️ 先做三件事

1. **把底盘架空**（垫两本书让三个轮子悬空），或者放在空旷地面上。第一次跑方向大概率是错的。
2. 确认舵机驱动板 USB 插在 Orin 上、12V 已上电。
3. 手放在电源开关上。任何时候想停 → 跑最后那个 **急停** 格子。

## 这个 notebook 要回答的问题

编号 7/8/9 是我们按「随手拿起哪个舵机」的顺序写进去的，所以**哪个 ID 是哪个物理轮子，现在还不知道**。
而 lerobot 的运动学**写死了对应关系**（已对着 v0.5.2 源码逐项验证，`base_move.py` 与它零误差）：

| ID | lerobot 里的名字 | 安装角 |
|---|---|---|
| **7** | `base_left_wheel` — **左轮** | 150° |
| **8** | `base_back_wheel` — **后轮** | -90° |
| **9** | `base_right_wheel` — **右轮** | 30° |

所以先跑 **第一步** 辨认三个轮子，对不上再调整 —— 别急着跑第二步。

## 0. 体检：9 个电机都在吗

In [ ]:
!ssh jatson@192.168.3.188 'ls -l /dev/ttyACM* /dev/ttyUSB* 2>&1 | head -3'
!ssh jatson@192.168.3.188 'python3 ~/scan_servos.py /dev/ttyACM0'

期望：`ID 1~9` 全部应答，电压 11~12V。

少了谁 → 那一段线没接好。全都没有 → 驱动板没插或没上电。

---
## 第一步：辨认三个轮子 / Identify the wheels

一次只转一个轮子。**记下转的是哪个轮子**（左前 / 右前 / 后，或者你自己的叫法）。

In [ ]:
# 7 号轮 —— 哪个轮子转了？
!ssh jatson@192.168.3.188 'python3 ~/base_move.py /dev/ttyACM0 wheel:7 0.8'

In [ ]:
# 8 号轮
!ssh jatson@192.168.3.188 'python3 ~/base_move.py /dev/ttyACM0 wheel:8 0.8'

In [ ]:
# 9 号轮
!ssh jatson@192.168.3.188 'python3 ~/base_move.py /dev/ttyACM0 wheel:9 0.8'

**把结果记在这里**（双击编辑）：

| ID | lerobot 期望 | 实际是哪个轮子？ |
|---|---|---|
| 7 | **左轮** `base_left_wheel` | ? |
| 8 | **后轮** `base_back_wheel` | ? |
| 9 | **右轮** `base_right_wheel` | ? |

对照组装文档里的编号图，如果 ID 和安装位置对不上，有两条路：

- **改线**：把舵机线换个插法（最笨最清晰）。
- **改号**：`python3 ~/set_servo_id.py /dev/ttyACM0 <旧号> <新号>` —— 但**一次只能接一个舵机**，否则脚本会拒绝执行。
**最灵的一个判据**：跑「前进」时，运动学算出后轮(8)速度是 **0** —— 如果 8 号动了，或者某个轮子该转不转，就是映射错了。


---
## 第二步：底盘运动 / Base motion

三个轮子按全向运动学协同。参数：`<动作> <时长秒> <倍率>`，默认 `0.6 秒 / 倍率 1.0` ≈ 走 5cm 或转 18°。

**先架空跑一遍看方向对不对，再落地。**

In [ ]:
# 前进一点点
!ssh jatson@192.168.3.188 'python3 ~/base_move.py /dev/ttyACM0 forward 0.6'

In [ ]:
# 后退一点点
!ssh jatson@192.168.3.188 'python3 ~/base_move.py /dev/ttyACM0 back 0.6'

In [ ]:
# 左转一点点（逆时针）
!ssh jatson@192.168.3.188 'python3 ~/base_move.py /dev/ttyACM0 turn_left 0.6'

In [ ]:
# 右转一点点（顺时针）
!ssh jatson@192.168.3.188 'python3 ~/base_move.py /dev/ttyACM0 turn_right 0.6'

### 全向底盘的加分项：横着走（不转身）

In [ ]:
# 左移一点点（车头不转，整体平移）
!ssh jatson@192.168.3.188 'python3 ~/base_move.py /dev/ttyACM0 left 0.6'

In [ ]:
# 右移一点点
!ssh jatson@192.168.3.188 'python3 ~/base_move.py /dev/ttyACM0 right 0.6'

---
## 🛑 急停 / Emergency stop

三个轮子速度清零 + 松扭矩。任何时候都能跑。

In [ ]:
!ssh jatson@192.168.3.188 'python3 ~/base_move.py /dev/ttyACM0 stop'

---
## 方向不对怎么办 / When directions are wrong

这是**预期内**的，因为 ID 和物理安装位的对应关系还没验证过。按症状对号入座：

| 症状 | 原因 | 怎么修 |
|---|---|---|
| 「前进」变成原地打转/乱扭 | ID ↔ 轮子位置对不上 | 按第一步的结果重排：左轮改 7、后轮改 8、右轮改 9 |
| 「前进」变后退（其余都对） | 整体符号反 | 改 `base_move.py` 里 `MOTIONS` 表中 `forward` 那行的符号 |
| 「左转」变右转 | 旋转符号反 | 同上，改 `turn_left` / `turn_right` |
| 某个轮子完全不动 | 那个舵机没进速度模式 / 线松了 | 重跑第 0 格扫描，看它还在不在 |
| 轮子转了但车不动 | 机械问题：舵盘/联轴器/轮毂没锁紧 | 拧紧。回读的位置是真实编码器值，电气上舵机确实转了 |

**别在手柄层面拿符号硬补方向** —— 底层映射错了就在底层改，否则录数据、跑策略时会以更难查的方式再咬你一口。

## 参数怎么调

```bash
python3 ~/base_move.py /dev/ttyACM0 forward 1.2      # 走久一点（1.2 秒）
python3 ~/base_move.py /dev/ttyACM0 forward 0.6 0.5  # 慢一半
python3 ~/base_move.py /dev/ttyACM0 turn_left 0.6 2  # 转快一倍
```

## 附：单个舵机（机械臂也能用）

位置模式，转固定角度再回来。`4096 counts = 360°`。

```bash
python3 ~/move_servo2.py /dev/ttyACM0 <ID> <counts> <速度>
```

⚠️ **机械臂(1~6)先别乱试** —— 没标定、没限位，容易自己撞自己。等标定完再动。

In [ ]:
# 例：9 号轮位置模式转 180°（注意：这会把它切回位置模式的语义，
# 但轮子现在是速度模式，位置指令无效 —— 想用这个先把 mode 改回 0）
# !ssh jatson@192.168.3.188 'python3 ~/move_servo2.py /dev/ttyACM0 9 2048 1500'
